# T4.2 Accuracy Testing - Golden Set 100 STEM Questions

**Objective:** Generate 100 STEM quiz questions via pipeline, then evaluate correctness using LLM evaluator.

**Target:** ≥ 98% accuracy

**Subjects:** Math, Physics, Chemistry (Grades 10-12)

**Method:**
1. Generate 10 quiz questions per topic (10 topics = 100 questions)
2. Evaluate each question with Gemini 2.5 Flash (independent evaluator)
3. Score: correct_answer, explanation_accuracy, overall_pass
4. Report accuracy % and failure analysis

## 0. Setup

In [1]:
import sys, time, re, json, asyncio
from pathlib import Path
from IPython.display import display, Markdown, HTML

project_root = str(Path.cwd().parent.parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.config import get_settings
from src.api.schemas import (
    GenerationRequest, GameType, DifficultyLevel, DocScope,
    GameContentResponse, QuizQuestion,
)
from src.graph.builder import compile_graph
from src.graph.state import MAX_REVIEW_ITERATIONS
from src.services.llm import get_generation_llm

settings = get_settings()

display(Markdown(f"""
### Configuration
| Setting | Value |
|---------|-------|
| GCP Project | `{settings.gcp_project_id}` |
| Generation Model | `{settings.generation_model}` @ `{settings.generation_model_location or settings.gcp_location}` |
| Review Model | `{settings.review_model}` @ `{settings.review_model_location}` |
| Max Review Iterations | `{MAX_REVIEW_ITERATIONS}` |
"""))
print('Setup OK ✓')

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



### Configuration
| Setting | Value |
|---------|-------|
| GCP Project | `green-mercury-485016-n1` |
| Generation Model | `gemini-2.5-flash` @ `asia-southeast1` |
| Review Model | `gemini-3.1-flash-lite-preview` @ `global` |
| Max Review Iterations | `3` |


Setup OK ✓


## 1. Golden Test Set Definition

In [2]:
# 10 topics x 10 questions = 100 questions
# Covering Math, Physics, Chemistry across grades 10-12
GOLDEN_TOPICS = [
    # Math (4 topics)
    {'topic': 'Hàm số bậc hai và đồ thị', 'subject': 'Toán 10', 'difficulty': 'comprehension'},
    {'topic': 'Đạo hàm và ứng dụng của đạo hàm', 'subject': 'Toán 11', 'difficulty': 'application'},
    {'topic': 'Tích phân và ứng dụng', 'subject': 'Toán 12', 'difficulty': 'comprehension'},
    {'topic': 'Số phức và phương trình bậc hai trong tập số phức', 'subject': 'Toán 12', 'difficulty': 'application'},
    # Physics (3 topics)
    {'topic': 'Chuyển động thẳng đều và chuyển động thẳng biến đổi đều', 'subject': 'Vật lý 10', 'difficulty': 'comprehension'},
    {'topic': 'Các định luật Newton về chuyển động', 'subject': 'Vật lý 10', 'difficulty': 'application'},
    {'topic': 'Dao động điều hòa', 'subject': 'Vật lý 12', 'difficulty': 'comprehension'},
    # Chemistry (3 topics)
    {'topic': 'Bảng tuần hoàn các nguyên tố hóa học và quy luật biến đổi tuần hoàn', 'subject': 'Hóa học 10', 'difficulty': 'recall'},
    {'topic': 'Phản ứng oxi hóa khử và cân bằng phương trình', 'subject': 'Hóa học 10', 'difficulty': 'application'},
    {'topic': 'Este và chất béo', 'subject': 'Hóa học 12', 'difficulty': 'comprehension'},
]

NUM_QUESTIONS_PER_TOPIC = 10
TOTAL_EXPECTED = len(GOLDEN_TOPICS) * NUM_QUESTIONS_PER_TOPIC

display(Markdown(f"""
### Golden Test Set
| # | Subject | Topic | Difficulty | Questions |
|---|---------|-------|------------|-----------|
{''.join(f'| {i+1} | {t["subject"]} | {t["topic"]} | {t["difficulty"]} | {NUM_QUESTIONS_PER_TOPIC} |' + chr(10) for i, t in enumerate(GOLDEN_TOPICS))}
| | | **TOTAL** | | **{TOTAL_EXPECTED}** |
"""))
print(f'{len(GOLDEN_TOPICS)} topics, {TOTAL_EXPECTED} questions expected')


### Golden Test Set
| # | Subject | Topic | Difficulty | Questions |
|---|---------|-------|------------|-----------|
| 1 | Toán 10 | Hàm số bậc hai và đồ thị | comprehension | 10 |
| 2 | Toán 11 | Đạo hàm và ứng dụng của đạo hàm | application | 10 |
| 3 | Toán 12 | Tích phân và ứng dụng | comprehension | 10 |
| 4 | Toán 12 | Số phức và phương trình bậc hai trong tập số phức | application | 10 |
| 5 | Vật lý 10 | Chuyển động thẳng đều và chuyển động thẳng biến đổi đều | comprehension | 10 |
| 6 | Vật lý 10 | Các định luật Newton về chuyển động | application | 10 |
| 7 | Vật lý 12 | Dao động điều hòa | comprehension | 10 |
| 8 | Hóa học 10 | Bảng tuần hoàn các nguyên tố hóa học và quy luật biến đổi tuần hoàn | recall | 10 |
| 9 | Hóa học 10 | Phản ứng oxi hóa khử và cân bằng phương trình | application | 10 |
| 10 | Hóa học 12 | Este và chất béo | comprehension | 10 |

| | | **TOTAL** | | **100** |


10 topics, 100 questions expected


## 2. Helper Functions

In [3]:
def make_request(topic: str, difficulty: str = 'comprehension', num_questions: int = 10) -> GenerationRequest:
    return GenerationRequest(
        user_id='accuracy_test_t42',
        topic=topic,
        game_types=[GameType('quiz')],
        num_questions=num_questions,
        difficulty=DifficultyLevel(difficulty),
        doc_scope=DocScope('system'),
    )

async def run_pipeline(request: GenerationRequest) -> dict:
    app = compile_graph()
    initial_state = {
        'request': request,
        'doc_scope': request.doc_scope.value,
        'iteration_count': 0,
        'rejected_items': [],
        'errors': [],
    }
    t0 = time.time()
    result = await app.ainvoke(initial_state)
    elapsed = time.time() - t0
    return {**result, '_elapsed': elapsed}

print('Helpers loaded ✓')

Helpers loaded ✓


## 3. Generate All Questions

In [4]:
all_questions: list[dict] = []  # {'topic_info': {...}, 'quiz': QuizQuestion, 'topic_idx': int}
generation_stats = []

for idx, topic_info in enumerate(GOLDEN_TOPICS):
    print(f'\n[{idx+1}/{len(GOLDEN_TOPICS)}] Generating: {topic_info["subject"]} - {topic_info["topic"]}...')
    req = make_request(topic_info['topic'], difficulty=topic_info['difficulty'])
    
    try:
        result = await run_pipeline(req)
        output: GameContentResponse | None = result.get('final_output')
        elapsed = result['_elapsed']
        errors = result.get('errors', [])
        iterations = result.get('iteration_count', 0)
        
        if output is None:
            print(f'  ❌ No output! Errors: {errors}')
            generation_stats.append({
                'topic': topic_info['topic'], 'subject': topic_info['subject'],
                'count': 0, 'elapsed': elapsed, 'errors': errors, 'iterations': iterations,
            })
            continue
        
        quizzes = output.content.quiz
        for q in quizzes:
            all_questions.append({'topic_info': topic_info, 'quiz': q, 'topic_idx': idx})
        
        meta = output.metadata
        generation_stats.append({
            'topic': topic_info['topic'], 'subject': topic_info['subject'],
            'count': len(quizzes), 'elapsed': elapsed,
            'generated': meta.total_generated, 'passed': meta.total_passed_review,
            'rejected': meta.total_rejected, 'iterations': iterations,
        })
        print(f'  ✅ {len(quizzes)} questions in {elapsed:.0f}s (gen: {meta.total_generated}, pass: {meta.total_passed_review})')
    except Exception as e:
        print(f'  ❌ Exception: {e}')
        generation_stats.append({
            'topic': topic_info['topic'], 'subject': topic_info['subject'],
            'count': 0, 'elapsed': 0, 'errors': [str(e)], 'iterations': 0,
        })

total_generated = len(all_questions)
total_time = sum(s['elapsed'] for s in generation_stats)
print(f'\n=== Generation Complete: {total_generated}/{TOTAL_EXPECTED} questions in {total_time:.0f}s ===')


[1/10] Generating: Toán 10 - Hàm số bậc hai và đồ thị...
2026-03-22 16:37:51 [info     ] creating_graph                
2026-03-22 16:37:51 [info     ] graph_created                 
2026-03-22 16:37:51 [info     ] graph_compiled_without_checkpointer
2026-03-22 16:37:51 [info     ] supervisor_analyzing           doc_scope=system topic='Hàm số bậc hai và đồ thị' user_id=accuracy_test_t42
2026-03-22 16:37:51 [debug    ] creating_review_llm            location=global max_output_tokens=10 model=gemini-3.1-flash-lite-preview temperature=0


/home/sakana/Code/ManifoldsTeam/AIServices/src/services/llm.py:99: LangChainDeprecationWarning: The class `ChatVertexAI` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import ChatGoogleGenerativeAI``.
  return ChatVertexAI(


2026-03-22 16:37:53 [info     ] supervisor_classified          content_type=math
2026-03-22 16:37:53 [info     ] routing_to_agent               content_type=math route=math_agent
2026-03-22 16:37:53 [info     ] math_agent_starting            doc_scope=system num_questions=10 num_to_generate=13 topic='Hàm số bậc hai và đồ thị' user_id=accuracy_test_t42
2026-03-22 16:37:53 [info     ] retrieving_context             doc_scope=system query='Hàm số bậc hai và đồ thị comprehension level educational content' user_id=accuracy_test_t42
2026-03-22 16:37:53 [debug    ] creating_retriever             doc_scope=system extractive_answers=True filter='user_id: ANY("__system__")' search_engine_id=gp-mathagent_1773042630372 user_id=accuracy_test_t42
2026-03-22 16:37:53 [debug    ] retrieving_documents           doc_scope=system query='Hàm số bậc hai và đồ thị comprehension level educational content'


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-22 16:37:55 [debug    ] retrieved_documents            count=27
2026-03-22 16:37:55 [info     ] context_retrieved              chunks=27 sources=27
2026-03-22 16:37:55 [info     ] math_agent_context_retrieved   context_chunks=27
2026-03-22 16:37:55 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.7
2026-03-22 16:38:36 [info     ] math_agent_code_execution_done code_traces_count=6 text_length=12337
2026-03-22 16:38:36 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.3
2026-03-22 16:38:48 [info     ] math_agent_batch_parsed        batch=1 items=7
2026-03-22 16:39:00 [info     ] math_agent_batch_parsed        batch=2 items=6
2026-03-22 16:39:00 [info     ] math_agent_generated           items_count=13
2026-03-22 16:39:00 [info     ] reviewer_reviewing             items_count=13 iteration=0
2026-03-22 16:39:00 [debug    ] crea

...

2026-03-22 17:04:43 [debug    ] retrieved_documents            count=24
2026-03-22 17:04:43 [info     ] context_retrieved              chunks=24 sources=24
2026-03-22 17:04:43 [info     ] math_agent_context_retrieved   context_chunks=24
2026-03-22 17:04:43 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.7
2026-03-22 17:07:05 [info     ] math_agent_code_execution_done code_traces_count=48 text_length=45115
2026-03-22 17:07:05 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.3
2026-03-22 17:07:27 [warning  ] math_agent_batch_parse_failed  batch=1
2026-03-22 17:07:52 [warning  ] math_agent_batch_parse_failed  batch=2
2026-03-22 17:07:52 [error    ] math_agent_structured_output_all_batches_failed
2026-03-22 17:07:52 [warning  ] reviewer_no_items_to_review   
2026-03-22 17:07:52 [info     ] review_router_deciding         iteration=1 m

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-22 17:07:56 [debug    ] retrieved_documents            count=24
2026-03-22 17:07:56 [info     ] context_retrieved              chunks=24 sources=24
2026-03-22 17:07:56 [info     ] math_agent_context_retrieved   context_chunks=24
2026-03-22 17:07:56 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.7
2026-03-22 17:09:11 [info     ] math_agent_code_execution_done code_traces_count=12 text_length=24379
2026-03-22 17:09:11 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.3
2026-03-22 17:09:32 [info     ] math_agent_batch_parsed        batch=1 items=7
2026-03-22 17:09:48 [info     ] math_agent_batch_parsed        batch=2 items=6
2026-03-22 17:09:48 [info     ] math_agent_generated           items_count=13
2026-03-22 17:09:48 [info     ] reviewer_reviewing             items_count=13 iteration=1
2026-03-22 17:09:48 [debug    ] cre

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-22 17:10:11 [debug    ] retrieved_documents            count=30
2026-03-22 17:10:11 [info     ] context_retrieved              chunks=30 sources=30
2026-03-22 17:10:11 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-22 17:10:11 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.7
2026-03-22 17:10:45 [info     ] math_agent_code_execution_done code_traces_count=2 text_length=11701
2026-03-22 17:10:45 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.3
2026-03-22 17:10:55 [info     ] math_agent_batch_parsed        batch=1 items=7
2026-03-22 17:11:07 [info     ] math_agent_batch_parsed        batch=2 items=6
2026-03-22 17:11:07 [info     ] math_agent_generated           items_count=13
2026-03-22 17:11:07 [info     ] reviewer_reviewing             items_count=13 iteration=0
2026-03-22 17:11:07 [debug    ] crea

In [5]:
# Generation summary table
rows = ''
for s in generation_stats:
    status = '✅' if s['count'] >= NUM_QUESTIONS_PER_TOPIC else '⚠️' if s['count'] > 0 else '❌'
    rows += f"| {status} | {s['subject']} | {s['topic'][:40]} | {s['count']}/{NUM_QUESTIONS_PER_TOPIC} | {s.get('generated','?')}→{s.get('passed','?')} | {s['iterations']} | {s['elapsed']:.0f}s |\n"

display(Markdown(f"""
### Generation Results
| | Subject | Topic | Yield | Gen→Pass | Iter | Time |
|---|---------|-------|-------|----------|------|------|
{rows}
| | | **TOTAL** | **{total_generated}/{TOTAL_EXPECTED}** | | | **{total_time:.0f}s** |
"""))


### Generation Results
| | Subject | Topic | Yield | Gen→Pass | Iter | Time |
|---|---------|-------|-------|----------|------|------|
| ✅ | Toán 10 | Hàm số bậc hai và đồ thị | 10/10 | 10→10 | 1 | 88s |
| ✅ | Toán 11 | Đạo hàm và ứng dụng của đạo hàm | 10/10 | 10→10 | 1 | 208s |
| ✅ | Toán 12 | Tích phân và ứng dụng | 10/10 | 10→10 | 1 | 118s |
| ✅ | Toán 12 | Số phức và phương trình bậc hai trong tậ | 10/10 | 11→10 | 3 | 813s |
| ✅ | Vật lý 10 | Chuyển động thẳng đều và chuyển động thẳ | 10/10 | 10→10 | 1 | 95s |
| ✅ | Vật lý 10 | Các định luật Newton về chuyển động | 10/10 | 10→10 | 1 | 166s |
| ✅ | Vật lý 12 | Dao động điều hòa | 10/10 | 10→10 | 1 | 69s |
| ✅ | Hóa học 10 | Bảng tuần hoàn các nguyên tố hóa học và  | 10/10 | 10→10 | 1 | 52s |
| ✅ | Hóa học 10 | Phản ứng oxi hóa khử và cân bằng phương  | 10/10 | 11→10 | 2 | 328s |
| ✅ | Hóa học 12 | Este và chất béo | 10/10 | 10→10 | 1 | 83s |

| | | **TOTAL** | **100/100** | | | **2019s** |


## 4. LLM Accuracy Evaluator

Use Gemini 2.5 Flash as an independent evaluator to verify each question's correctness.

In [17]:
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage

class AccuracyVerdict(BaseModel):
    """Evaluator output for a single quiz question."""
    correct_answer_valid: bool = Field(description='Is the marked correct answer actually correct?')
    explanation_accurate: bool = Field(description='Is the explanation factually accurate?')
    distractors_plausible: bool = Field(description='Are wrong options plausible (not obviously wrong)?')
    reasoning: str = Field(description='Brief reasoning for the verdict (1-2 sentences)')

EVALUATOR_SYSTEM = """You are an expert Vietnamese STEM education evaluator.
You verify the correctness of quiz questions for Vietnamese high school students (THPT).

For each quiz question, verify:
1. **Correct answer**: Is the option marked as correct ACTUALLY the right answer?
   - For math: verify computations step by step
   - For physics: check formulas, units, and numerical results
   - For chemistry: check reactions, properties, and nomenclature
2. **Explanation**: Is the explanation factually accurate and consistent with the correct answer?
3. **Distractors**: Are the wrong options plausible enough to be educational?

Be strict on correctness. A wrong answer or wrong explanation = FAIL.
Minor Vietnamese language issues are OK if the content is correct.
Keep reasoning brief (1-2 sentences max)."""

def format_quiz_for_eval(q: QuizQuestion) -> str:
    options = '\n'.join(
        f"  {'[CORRECT]' if o.is_correct else '[WRONG]'} {o.text}"
        for o in q.options
    )
    trace = f'\nComputation trace: {q.computation_trace[:500]}' if q.computation_trace else ''
    return f"""Topic: {q.topic}
Difficulty: {q.difficulty}
Question: {q.question}
Options:
{options}
Explanation: {q.explanation}{trace}"""

# Increase max_output_tokens to 4096 to prevent truncation (was 1024)
eval_llm = get_generation_llm(temperature=0.0, max_output_tokens=4096)
eval_structured = eval_llm.with_structured_output(AccuracyVerdict)

print(f'Evaluator ready: {settings.generation_model} @ temp=0.0, max_tokens=4096')

2026-03-22 17:53:22 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=4096 model=gemini-2.5-flash temperature=0.0
Evaluator ready: gemini-2.5-flash @ temp=0.0, max_tokens=4096


In [18]:
# Evaluate all questions with retry logic for structured output failures
eval_results: list[dict] = []
EVAL_BATCH_PAUSE = 0.5   # seconds between calls
EVAL_MAX_RETRIES = 3     # retry up to 3 times on None verdict

print(f'Evaluating {len(all_questions)} questions...')
eval_t0 = time.time()

for i, item in enumerate(all_questions):
    q: QuizQuestion = item['quiz']
    prompt = format_quiz_for_eval(q)
    
    verdict = None
    for attempt in range(EVAL_MAX_RETRIES):
        try:
            verdict = await eval_structured.ainvoke([
                SystemMessage(content=EVALUATOR_SYSTEM),
                HumanMessage(content=f'Evaluate this quiz question:\n\n{prompt}'),
            ])
            if verdict is not None:
                break
            # structured output returned None — retry
            if attempt < EVAL_MAX_RETRIES - 1:
                print(f'  [{i+1}] ⚠️ None result, retrying ({attempt+2}/{EVAL_MAX_RETRIES})...')
                await asyncio.sleep(1.0 * (attempt + 1))
        except Exception as e:
            if attempt < EVAL_MAX_RETRIES - 1:
                print(f'  [{i+1}] ⚠️ Error: {e}, retrying ({attempt+2}/{EVAL_MAX_RETRIES})...')
                await asyncio.sleep(1.0 * (attempt + 1))
            else:
                print(f'  [{i+1}] ❌ Eval failed after {EVAL_MAX_RETRIES} attempts: {e}')
    
    eval_results.append({'question_idx': i, 'verdict': verdict, 'quiz': q, 'topic_info': item['topic_info']})
    
    if verdict is not None:
        status = '✅' if verdict.correct_answer_valid and verdict.explanation_accurate else '❌'
        if (i + 1) % 10 == 0 or not (verdict.correct_answer_valid and verdict.explanation_accurate):
            print(f'  [{i+1}/{len(all_questions)}] {status} {q.topic[:30]}... | ans={verdict.correct_answer_valid} exp={verdict.explanation_accurate}')
    else:
        print(f'  [{i+1}/{len(all_questions)}] ⚠️ STILL None after {EVAL_MAX_RETRIES} retries')
    
    if EVAL_BATCH_PAUSE > 0:
        await asyncio.sleep(EVAL_BATCH_PAUSE)

eval_time = time.time() - eval_t0
eval_errors_count = sum(1 for r in eval_results if r['verdict'] is None)
print(f'\nEvaluation complete in {eval_time:.0f}s')
print(f'Evaluated: {len(eval_results) - eval_errors_count}/{len(eval_results)} | Errors: {eval_errors_count}')

Evaluating 100 questions...
  [10/100] ✅ Miền xác định và miền giá trị... | ans=True exp=True
  [20/100] ✅ Ứng dụng đạo hàm - Tính đơn đi... | ans=True exp=True
  [30/100] ✅ Định lý Newton-Leibniz... | ans=True exp=True
  [40/100] ✅ Phương trình chứa số phức và s... | ans=True exp=True
  [50/100] ✅ So sánh các loại chuyển động... | ans=True exp=True
  [60/100] ✅ Định luật II Newton và chuyển ... | ans=True exp=True
  [70/100] ✅ Dao động điều hòa - Mối liên h... | ans=True exp=True
  [80/100] ✅ Mối quan hệ giữa cấu tạo nguyê... | ans=True exp=True
  [90/100] ✅ Phản ứng oxi hóa-khử, Bài toán... | ans=True exp=True
  [100/100] ✅ Tính toán phản ứng xà phòng hó... | ans=True exp=True

Evaluation complete in 334s
Evaluated: 100/100 | Errors: 0


## 5. Accuracy Analysis

In [19]:
# Overall accuracy
total_evaluated = len([r for r in eval_results if r['verdict'] is not None])
correct_answer = sum(1 for r in eval_results if r['verdict'] and r['verdict'].correct_answer_valid)
correct_explanation = sum(1 for r in eval_results if r['verdict'] and r['verdict'].explanation_accurate)
correct_distractors = sum(1 for r in eval_results if r['verdict'] and r['verdict'].distractors_plausible)
fully_correct = sum(1 for r in eval_results if r['verdict'] and r['verdict'].correct_answer_valid and r['verdict'].explanation_accurate)

ans_pct = correct_answer / total_evaluated * 100 if total_evaluated > 0 else 0
exp_pct = correct_explanation / total_evaluated * 100 if total_evaluated > 0 else 0
dist_pct = correct_distractors / total_evaluated * 100 if total_evaluated > 0 else 0
full_pct = fully_correct / total_evaluated * 100 if total_evaluated > 0 else 0

target_met = '✅ TARGET MET' if full_pct >= 98 else '❌ BELOW TARGET'

display(Markdown(f"""
## Overall Accuracy Results

| Metric | Count | Percentage | Target |
|--------|-------|------------|--------|
| Total generated | {len(all_questions)} | | 100 |
| Total evaluated | {total_evaluated} | | |
| Correct answer | {correct_answer}/{total_evaluated} | **{ans_pct:.1f}%** | ≥ 98% |
| Accurate explanation | {correct_explanation}/{total_evaluated} | **{exp_pct:.1f}%** | ≥ 98% |
| Plausible distractors | {correct_distractors}/{total_evaluated} | **{dist_pct:.1f}%** | |
| **Fully correct (ans + exp)** | **{fully_correct}/{total_evaluated}** | **{full_pct:.1f}%** | **≥ 98%** |

### {target_met} — Accuracy: {full_pct:.1f}%
"""))


## Overall Accuracy Results

| Metric | Count | Percentage | Target |
|--------|-------|------------|--------|
| Total generated | 100 | | 100 |
| Total evaluated | 100 | | |
| Correct answer | 100/100 | **100.0%** | ≥ 98% |
| Accurate explanation | 100/100 | **100.0%** | ≥ 98% |
| Plausible distractors | 95/100 | **95.0%** | |
| **Fully correct (ans + exp)** | **100/100** | **100.0%** | **≥ 98%** |

### ✅ TARGET MET — Accuracy: 100.0%


In [20]:
# Per-subject breakdown
from collections import defaultdict
subject_stats = defaultdict(lambda: {'total': 0, 'correct': 0, 'ans_fail': 0, 'exp_fail': 0})

for r in eval_results:
    if r['verdict'] is None:
        continue
    subj = r['topic_info']['subject'].split()[0]  # 'Toán', 'Vật', 'Hóa'
    subject_stats[subj]['total'] += 1
    if r['verdict'].correct_answer_valid and r['verdict'].explanation_accurate:
        subject_stats[subj]['correct'] += 1
    if not r['verdict'].correct_answer_valid:
        subject_stats[subj]['ans_fail'] += 1
    if not r['verdict'].explanation_accurate:
        subject_stats[subj]['exp_fail'] += 1

rows = ''
for subj in ['Toán', 'Vật', 'Hóa']:
    s = subject_stats[subj]
    pct = s['correct'] / s['total'] * 100 if s['total'] > 0 else 0
    name = {'Toán': 'Mathematics', 'Vật': 'Physics', 'Hóa': 'Chemistry'}[subj]
    rows += f"| {name} | {s['total']} | {s['correct']} | {pct:.1f}% | {s['ans_fail']} | {s['exp_fail']} |\n"

display(Markdown(f"""
### Per-Subject Accuracy
| Subject | Evaluated | Correct | Accuracy | Ans Fails | Exp Fails |
|---------|-----------|---------|----------|-----------|----------|
{rows}"""))


### Per-Subject Accuracy
| Subject | Evaluated | Correct | Accuracy | Ans Fails | Exp Fails |
|---------|-----------|---------|----------|-----------|----------|
| Mathematics | 40 | 40 | 100.0% | 0 | 0 |
| Physics | 30 | 30 | 100.0% | 0 | 0 |
| Chemistry | 30 | 30 | 100.0% | 0 | 0 |


In [21]:
# Show ALL failures for debugging
failures = [r for r in eval_results if r['verdict'] and not (r['verdict'].correct_answer_valid and r['verdict'].explanation_accurate)]
eval_errors = [r for r in eval_results if r['verdict'] is None]

if not failures and not eval_errors:
    display(Markdown('### ✅ No failures detected!'))
else:
    display(Markdown(f'### ❌ {len(failures)} failures + {len(eval_errors)} eval errors\n'))
    for f in failures:
        q = f['quiz']
        v = f['verdict']
        correct_opt = next((o.text for o in q.options if o.is_correct), 'N/A')
        display(Markdown(f"""
---
**FAIL #{f['question_idx']+1}** [{f['topic_info']['subject']}] — {q.topic}

> {q.question}

Marked correct: _{correct_opt}_

| Check | Result |
|-------|--------|
| Answer correct | {'✅' if v.correct_answer_valid else '❌'} |
| Explanation accurate | {'✅' if v.explanation_accurate else '❌'} |
| Distractors plausible | {'✅' if v.distractors_plausible else '❌'} |

**Evaluator reasoning:** {v.reasoning}
"""))

### ✅ No failures detected!

## 6. Final Summary

In [22]:
display(Markdown(f"""
## T4.2 Accuracy Test Report

**Date:** {time.strftime('%Y-%m-%d %H:%M')}

| Metric | Value |
|--------|-------|
| Topics tested | {len(GOLDEN_TOPICS)} |
| Questions generated | {len(all_questions)} / {TOTAL_EXPECTED} |
| Questions evaluated | {total_evaluated} |
| **Answer accuracy** | **{ans_pct:.1f}%** |
| **Explanation accuracy** | **{exp_pct:.1f}%** |
| **Overall accuracy (ans+exp)** | **{full_pct:.1f}%** |
| Target | ≥ 98% |
| **Result** | **{target_met}** |
| Generation time | {total_time:.0f}s |
| Evaluation time | {eval_time:.0f}s |
| Total time | {total_time + eval_time:.0f}s |

### Failures: {len(failures)} | Eval errors: {len(eval_errors)}
"""))


## T4.2 Accuracy Test Report

**Date:** 2026-03-22 17:59

| Metric | Value |
|--------|-------|
| Topics tested | 10 |
| Questions generated | 100 / 100 |
| Questions evaluated | 100 |
| **Answer accuracy** | **100.0%** |
| **Explanation accuracy** | **100.0%** |
| **Overall accuracy (ans+exp)** | **100.0%** |
| Target | ≥ 98% |
| **Result** | **✅ TARGET MET** |
| Generation time | 2019s |
| Evaluation time | 334s |
| Total time | 2353s |

### Failures: 0 | Eval errors: 0
